# 01. THIẾT LẬP MÔI TRƯỜNG, HIỂU NGHIỆP VỤ & DỮ LIỆU THÔ
---
* **Khung phương pháp luận**: CRISP-DM (Pha 1: Business Understanding & Pha 2: Data Understanding)
* **Trọng tâm thực hiện**: Khảo sát nghiệp vụ rủi ro tín dụng (Basel II/III, IFRS 9), ma trận chi phí sai lầm bất đối xứng, yêu cầu giải thích SHAP và kiểm toán cấu trúc dữ liệu thô
---


---
## 1. THIẾT LẬP MÔI TRƯỜNG & ĐỊNH VỊ DỮ LIỆU

Khởi tạo runtime Python 3.10, cấu hình Pandas và thiết lập cơ chế định vị tệp dữ liệu thô (`data/raw`).


In [1]:
# 1.1. Nạp thư viện cốt lõi và kiểm tra phiên bản
import os
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

# Cấu hình hiển thị bảng Pandas
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', lambda x: '%.4f' % x)

# Cố định Random Seed để đảm bảo tính tái lập
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print('=' * 65)
print('KIỂM TRA CẤU HÌNH MÔI TRƯỜNG THỰC THI:')
print('=' * 65)
print(f'Python Executable : {sys.executable}')
print(f'Python Version    : {sys.version.split()[0]}')
print(f'NumPy Version     : {np.__version__}')
print(f'Pandas Version    : {pd.__version__}')
print('=' * 65)
print('[✓] Môi trường Python 3.10 đã sẵn sàng!')


KIỂM TRA CẤU HÌNH MÔI TRƯỜNG THỰC THI:
Python Executable : c:\Users\MSI\Downloads\Specialized Prọect\.venv\Scripts\python.exe
Python Version    : 3.10.20
NumPy Version     : 1.26.4
Pandas Version    : 2.3.3
[✓] Môi trường Python 3.10 đã sẵn sàng!


In [2]:
# 1.2. Định vị đường dẫn dữ liệu thô linh hoạt
def find_data_file(filename):
    candidates = [
        os.path.join('data', 'raw', filename),
        os.path.join('..', 'data', 'raw', filename),
        os.path.join('credit-risk-xgb-shap', 'data', 'raw', filename),
        os.path.join('..', '..', 'data', 'raw', filename),
        os.path.join('data', filename)
    ]
    for path in candidates:
        if os.path.exists(path):
            return os.path.abspath(path)
    raise FileNotFoundError(f'Không tìm thấy file {filename} trong các đường dẫn tìm kiếm!')

primary_data_path = find_data_file('default_of_credit_card_clients.csv')
benchmark_data_path = find_data_file('south_german_credit.csv')

print(f'[*] File dữ liệu chính (Yeh & Lien 2009) : {primary_data_path}')
print(f'[*] File dữ liệu đối chứng (South German): {benchmark_data_path}')


[*] File dữ liệu chính (Yeh & Lien 2009) : c:\Users\MSI\Downloads\Specialized Prọect\credit-risk-xgb-shap\data\raw\default_of_credit_card_clients.csv
[*] File dữ liệu đối chứng (South German): c:\Users\MSI\Downloads\Specialized Prọect\credit-risk-xgb-shap\data\raw\south_german_credit.csv


---
## 2. HIỂU NGHIỆP VỤ (BUSINESS UNDERSTANDING)

### 2.1. Bản chất Bài toán Chấm điểm Tín dụng & Xác suất Vỡ nợ (PD)
* **Mục tiêu**: Ước lượng xác suất khách hàng không hoàn thành nghĩa vụ trả nợ trong chu kỳ 12 tháng tiếp theo ($PD = P(Y=1|X)$).
* **Chuẩn mực Basel II/III & IFRS 9**: Tổn thất tín dụng dự kiến ($ECL$) được xác định bởi:
  $$ECL = PD \times LGD \times EAD$$
  * **$PD$ (Probability of Default)**: Biến mục tiêu cần dự đoán ($y \in \{0, 1\}$).
  * **$LGD$ (Loss Given Default)**: Tỷ lệ tổn thất khi vỡ nợ (45% – 85% đối với thẻ tín dụng không tài sản bảo đảm).
  * **$EAD$ (Exposure at Default)**: Tổng dư nợ thực tế tại thời điểm phát sinh vỡ nợ.

### 2.2. Ma trận Chi phí Sai lầm Bất đối xứng
Trong thẩm định tín dụng, thiệt hại tài chính giữa hai loại sai số có sự chênh lệch rõ rệt:

| Loại sai số | Định nghĩa thống kê | Hệ quả nghiệp vụ ngân hàng | Chi phí tương đối |
|---|---|---|:---:|
| **False Positive (Type I)** | Khách tốt ($Y=0$) dự đoán là Vỡ nợ ($Y=1$) | Từ chối nhầm khách tốt; mất doanh thu lãi và phí dịch vụ. | **1x** (Thấp) |
| **False Negative (Type II)** | Khách vỡ nợ ($Y=1$) dự đoán là Tốt ($Y=0$) | Duyệt nhầm khách vỡ nợ; mất toàn bộ vốn gốc cho vay ($EAD \times LGD$). | **5x – 20x** (Nghiêm trọng) |

* **Hệ quả thiết kế mô hình**: Độ chính xác (Accuracy) tại ngưỡng cố định 0.5 không phản ánh đúng rủi ro tài chính. Bắt buộc sử dụng các thước đo phân tách rủi ro phi phụ thuộc ngưỡng: **ROC-AUC** (mục tiêu chính), **PR-AUC** và **Balanced Accuracy**.

### 2.3. Yêu cầu Pháp lý về Tính Giải thích được (XAI) & Độ ổn định SHAP
* **Yêu cầu pháp lý**: Các quy định như ECOA/FCRA (Mỹ) và GDPR Điều 22 (EU) bắt buộc tổ chức tài chính phải cung cấp lý do minh bạch (*Adverse Action Notice*) khi từ chối cấp tín dụng. Phương pháp TreeSHAP được lựa chọn nhằm phân rã dự đoán thành tổng đóng góp từng đặc trưng.
* **Vấn đề ổn định giải thích (SHAP Stability)**: Mô hình ensemble (như XGBoost) khi tối ưu đơn mục tiêu theo AUC thường tạo ra các cấu hình cây sâu, nhạy cảm với biến động dữ liệu giữa các fold/seed. Điều này làm đảo lộn thứ hạng quan trọng của các đặc trưng, gây rủi ro tuân thủ pháp lý.
* **Bài toán Tối ưu Đa mục tiêu**: Cân bằng đồng thời hiệu năng dự đoán và độ ổn định giải thích:
  $$\max_{\theta} \; F(\theta) = \left[ \text{ROC-AUC}_{CV}(\theta), \; \text{SHAP-Stability}(\theta) \right]$$
  trong đó độ ổn định là hệ số tương quan thứ hạng Spearman trung bình của Global TreeSHAP qua 5 fold lặp trên cùng Reference Set cố định.


---
## 3. HIỂU DỮ LIỆU THÔ (DATA UNDERSTANDING)

### 3.1. Nguồn gốc & Quy mô Dữ liệu Chính
* **Tập dữ liệu**: *Default of Credit Card Clients* (Yeh & Lien, 2009) từ UCI Machine Learning Repository.
* **Quy mô**: 30.000 chủ thẻ tín dụng tại Đài Loan (04/2005 – 09/2005); 24 biến đặc trưng và 1 biến mục tiêu `default_payment_next_month`.


In [3]:
# 3.1. Đọc dữ liệu thô và kiểm tra kích thước bộ nhớ
df_raw = pd.read_csv(primary_data_path)

# Chuẩn hóa tên cột mục tiêu nếu có dấu cách hoặc dấu chấm
for target_name in ['default.payment.next.month', 'default payment next month']:
    if target_name in df_raw.columns:
        df_raw.rename(columns={target_name: 'default_payment_next_month'}, inplace=True)

print(f'[*] Số lượng quan sát (dòng)   : {df_raw.shape[0]:,}')
print(f'[*] Số lượng thuộc tính (cột)   : {df_raw.shape[1]}')
print(f'[*] Dung lượng bộ nhớ (RAM)     : {df_raw.memory_usage().sum() / (1024**2):.2f} MB')


[*] Số lượng quan sát (dòng)   : 30,000
[*] Số lượng thuộc tính (cột)   : 25
[*] Dung lượng bộ nhớ (RAM)     : 5.72 MB


### 3.2. Từ điển Dữ liệu Nghiệp vụ (Data Dictionary)

25 thuộc tính trong bộ dữ liệu được phân chia thành 4 nhóm nghiệp vụ tín dụng:

| STT | Tên thuộc tính | Kiểu | Nhóm nghiệp vụ | Diễn giải ý nghĩa nghiệp vụ & Thang đo |
|:---:|---|:---:|---|---|
| 0 | `ID` | Định danh | Quản trị | Mã định danh duy nhất của chủ thẻ (loại bỏ khi huấn luyện mô hình). |
| 1 | `LIMIT_BAL` | Liên tục | Hạn mức | Hạn mức tín dụng được cấp (Đài tệ - NT dollar), bao gồm cả hạn mức bổ sung cho người thân. |
| 2 | `SEX` | Phân loại | Nhân khẩu | Giới tính: `1` = Nam; `2` = Nữ. |
| 3 | `EDUCATION` | Phân loại | Nhân khẩu | Trình độ học vấn: `1` = Sau ĐH; `2` = ĐH; `3` = Phổ thông; `4` = Khác; `0, 5, 6` = Chưa xác định. |
| 4 | `MARRIAGE` | Phân loại | Nhân khẩu | Tình trạng hôn nhân: `1` = Đã kết hôn; `2` = Độc thân; `3` = Khác; `0` = Chưa xác định. |
| 5 | `AGE` | Liên tục | Nhân khẩu | Độ tuổi của chủ thẻ (năm). |
| 6–11 | `PAY_0` đến `PAY_6` | Thứ bậc | Trả nợ | Trạng thái thanh toán từ tháng 9 lùi về tháng 4/2005:<br>• `-2`: Không dư nợ; `-1`: Trả đủ; `0`: Dùng tín dụng quay vòng.<br>• `1` – `8`: Trễ hạn từ 1 đến 8 tháng; `9`: Trễ hạn từ 9 tháng trở lên. |
| 12–17 | `BILL_AMT1` đến `BILL_AMT6` | Liên tục | Dư nợ | Số dư sao kê hàng tháng từ tháng 9 lùi về tháng 4/2005 (NT dollar). Giá trị âm thể hiện trả dư/hoàn tiền. |
| 18–23 | `PAY_AMT1` đến `PAY_AMT6` | Liên tục | Thanh toán | Số tiền đã thanh toán trong kỳ sao kê trước (NT dollar). |
| 24 | `default_payment_next_month` | Nhị phân | **Mục tiêu** | Trạng thái vỡ nợ tháng 10/2005: `1` = Vỡ nợ (Default); `0` = Không vỡ nợ (Non-default). |


In [4]:
# 3.3. Quan sát các quan sát thô đầu tiên và cuối cùng
print('--- 5 DÒNG ĐẦU TIÊN CỦA TẬP DỮ LIỆU THÔ ---')
display(df_raw.head(5))

print('--- 5 DÒNG CUỐI CÙNG CỦA TẬP DỮ LIỆU THÔ ---')
display(df_raw.tail(5))


--- 5 DÒNG ĐẦU TIÊN CỦA TẬP DỮ LIỆU THÔ ---


,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,BILL_AMT1,BILL_AMT2,BILL_AMT3,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default_payment_next_month
0,1,20000.0000,2,2,1,24,2,2,-1,-1,-2,-2,3913.0000,3102.0000,689.0000,0.0000,0.0000,0.0000,0.0000,689.0000,0.0000,0.0000,0.0000,0.0000,1
1,2,120000.0000,2,2,2,26,-1,2,0,0,0,2,2682.0000,1725.0000,2682.0000,3272.0000,3455.0000,3261.0000,0.0000,1000.0000,1000.0000,1000.0000,0.0000,2000.0000,1
2,3,90000.0000,2,2,2,34,0,0,0,0,0,0,29239.0000,14027.0000,13559.0000,14331.0000,14948.0000,15549.0000,1518.0000,1500.0000,1000.0000,1000.0000,1000.0000,5000.0000,0
3,4,50000.0000,2,2,1,37,0,0,0,0,0,0,46990.0000,48233.0000,49291.0000,28314.0000,28959.0000,29547.0000,2000.0000,2019.0000,1200.0000,1100.0000,1069.0000,1000.0000,0
4,5,50000.0000,1,2,1,57,-1,0,-1,0,0,0,8617.0000,5670.0000,35835.0000,20940.0000,19146.0000,19131.0000,2000.0000,36681.0000,10000.0000,9000.0000,689.0000,679.0000,0


--- 5 DÒNG CUỐI CÙNG CỦA TẬP DỮ LIỆU THÔ ---


,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,BILL_AMT1,BILL_AMT2,BILL_AMT3,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default_payment_next_month
29995,29996,220000.0000,1,3,1,39,0,0,0,0,0,0,188948.0000,192815.0000,208365.0000,88004.0000,31237.0000,15980.0000,8500.0000,20000.0000,5003.0000,3047.0000,5000.0000,1000.0000,0
29996,29997,150000.0000,1,3,2,43,-1,-1,-1,-1,0,0,1683.0000,1828.0000,3502.0000,8979.0000,5190.0000,0.0000,1837.0000,3526.0000,8998.0000,129.0000,0.0000,0.0000,0
29997,29998,30000.0000,1,2,2,37,4,3,2,-1,0,0,3565.0000,3356.0000,2758.0000,20878.0000,20582.0000,19357.0000,0.0000,0.0000,22000.0000,4200.0000,2000.0000,3100.0000,1
29998,29999,80000.0000,1,3,1,41,1,-1,0,0,0,-1,-1645.0000,78379.0000,76304.0000,52774.0000,11855.0000,48944.0000,85900.0000,3409.0000,1178.0000,1926.0000,52964.0000,1804.0000,1
29999,30000,50000.0000,1,2,1,46,0,0,0,0,0,0,47929.0000,48905.0000,49764.0000,36535.0000,32428.0000,15313.0000,2078.0000,1800.0000,1430.0000,1000.0000,1000.0000,1000.0000,1


In [5]:
# 3.4. Kiểm tra cấu trúc lược đồ và kiểu dữ liệu
print('--- TỔNG HỢP KIỂU DỮ LIỆU CÁC CỘT (df.info()) ---')
df_raw.info()


--- TỔNG HỢP KIỂU DỮ LIỆU CÁC CỘT (df.info()) ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 25 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   ID                          30000 non-null  int64  
 1   LIMIT_BAL                   30000 non-null  float64
 2   SEX                         30000 non-null  int64  
 3   EDUCATION                   30000 non-null  int64  
 4   MARRIAGE                    30000 non-null  int64  
 5   AGE                         30000 non-null  int64  
 6   PAY_0                       30000 non-null  int64  
 7   PAY_2                       30000 non-null  int64  
 8   PAY_3                       30000 non-null  int64  
 9   PAY_4                       30000 non-null  int64  
 10  PAY_5                       30000 non-null  int64  
 11  PAY_6                       30000 non-null  int64  
 12  BILL_AMT1                   30000 non-

In [6]:
# 3.5. Kiểm toán sơ bộ tính toàn vẹn và chất lượng dữ liệu
print('=' * 65)
print('BÁO CÁO KIỂM TOÁN TÍNH TOÀN VẸN DỮ LIỆU:')
print('=' * 65)

total_nulls = df_raw.isnull().sum().sum()
total_duplicates = df_raw.duplicated().sum()
unique_ids = df_raw['ID'].nunique()

print(f'1. Tổng số giá trị bị thiếu (Null / NaN) : {total_nulls}')
print(f'2. Tổng số dòng bị trùng lặp hoàn toàn    : {total_duplicates}')
print(f'3. Số lượng mã ID tài khoản riêng biệt    : {unique_ids:,} (Khớp tổng số dòng: {unique_ids == len(df_raw)})')
print('=' * 65)

# Kiểm tra các tập giá trị phân loại để nhận diện nhãn dị biệt
print('\nTập các giá trị xuất hiện trong EDUCATION :',
      sorted(df_raw['EDUCATION'].unique().tolist()))
print('Tập các giá trị xuất hiện trong MARRIAGE  :',
      sorted(df_raw['MARRIAGE'].unique().tolist()))


BÁO CÁO KIỂM TOÁN TÍNH TOÀN VẸN DỮ LIỆU:
1. Tổng số giá trị bị thiếu (Null / NaN) : 0
2. Tổng số dòng bị trùng lặp hoàn toàn    : 0
3. Số lượng mã ID tài khoản riêng biệt    : 30,000 (Khớp tổng số dòng: True)

Tập các giá trị xuất hiện trong EDUCATION : [0, 1, 2, 3, 4, 5, 6]
Tập các giá trị xuất hiện trong MARRIAGE  : [0, 1, 2, 3]


### 3.6. Phân tích Biến Mục Tiêu (Target Variable Analysis)

Biến mục tiêu **`default_payment_next_month`** có hai trạng thái nhãn ($y \in \{0, 1\}$):
* **Lớp 0 (Non-default / Khách tốt)**: Thanh toán đầy đủ nghĩa vụ nợ trong tháng 10/2005.
* **Lớp 1 (Default / Khách vỡ nợ)**: Phát sinh nợ quá hạn trong tháng 10/2005.

**Đặc điểm phân phối & Ràng buộc kỹ thuật:**
1. **Mất cân bằng lớp**: Tỷ lệ vỡ nợ ~22.12% (tỷ số ~3.52:1). Phép chia tập và Cross-Validation bắt buộc dùng **Stratified Sampling** (`stratify=y`).
2. **Thước đo đánh giá**: Sử dụng ROC-AUC, PR-AUC và Balanced Accuracy thay thế cho Accuracy truyền thống.


In [7]:
# 3.6. Tính toán phân phối và tỷ lệ mất cân bằng của Biến mục tiêu
target_col = 'default_payment_next_month'
target_counts = df_raw[target_col].value_counts()
target_percentages = df_raw[target_col].value_counts(normalize=True) * 100

# Bảng tổng hợp phân tích biến mục tiêu
target_analysis_df = pd.DataFrame({
    'Ý nghĩa nghiệp vụ': ['Không vỡ nợ (Non-default)', 'Vỡ nợ (Default)'],
    'Giá trị nhãn (y)': [0, 1],
    'Số lượng (hồ sơ)': target_counts.values,
    'Tỷ lệ (%)': target_percentages.values,
    'Tỷ số đối ứng': ['1 : 1.00', f'1 : {target_counts[0]/target_counts[1]:.2f}']
})

print('=' * 70)
print('BẢNG PHÂN TÍCH BIẾN MỤC TIÊU (TARGET VARIABLE ANALYSIS):')
print('=' * 70)
display(target_analysis_df)
print('=' * 70)

imbalance_ratio = target_counts[0] / target_counts[1]
print(f'[*] Tổng số quan sát toàn bộ tập dữ liệu   : {len(df_raw):,} hồ sơ')
print(f'[*] Số lượng khách hàng Không vỡ nợ (Lớp 0): {target_counts[0]:,} hồ sơ ({target_percentages[0]:.2f}%)')
print(f'[*] Số lượng khách hàng Vỡ nợ (Lớp 1)      : {target_counts[1]:,} hồ sơ ({target_percentages[1]:.2f}%)')
print(f'[*] Tỷ số mất cân bằng lớp (Imbalance Ratio): {imbalance_ratio:.2f} : 1 (Cứ ~3.52 khách tốt thì có 1 khách vỡ nợ)')
print('\n[✓] Kết luận: Mất cân bằng ở mức vừa phải (22.12% Default). Cần dùng Stratified K-Fold khi chia tập.')


BẢNG PHÂN TÍCH BIẾN MỤC TIÊU (TARGET VARIABLE ANALYSIS):


,Ý nghĩa nghiệp vụ,Giá trị nhãn (y),Số lượng (hồ sơ),Tỷ lệ (%),Tỷ số đối ứng
0,Không vỡ nợ (Non-default),0,23364,77.8800,1 : 1.00
1,Vỡ nợ (Default),1,6636,22.1200,1 : 3.52


[*] Tổng số quan sát toàn bộ tập dữ liệu   : 30,000 hồ sơ
[*] Số lượng khách hàng Không vỡ nợ (Lớp 0): 23,364 hồ sơ (77.88%)
[*] Số lượng khách hàng Vỡ nợ (Lớp 1)      : 6,636 hồ sơ (22.12%)
[*] Tỷ số mất cân bằng lớp (Imbalance Ratio): 3.52 : 1 (Cứ ~3.52 khách tốt thì có 1 khách vỡ nợ)

[✓] Kết luận: Mất cân bằng ở mức vừa phải (22.12% Default). Cần dùng Stratified K-Fold khi chia tập.


### 3.7. Khảo sát Sơ bộ Dữ liệu Đối chứng (South German Credit)

Tập dữ liệu bổ trợ (1.000 quan sát, 21 thuộc tính) phục vụ đánh giá tính tổng quát hóa trên bối cảnh dữ liệu tín dụng khác biệt về quy mô và vùng địa lý.


In [8]:
# 3.7. Khảo sát sơ bộ Bộ dữ liệu đối chứng (South German Credit)
df_benchmark = pd.read_csv(benchmark_data_path)

print(f'[*] Kích thước South German Credit : {df_benchmark.shape[0]:,} dòng x {df_benchmark.shape[1]} cột')
print(f'[*] Tên cột mục tiêu đối chứng      : {df_benchmark.columns[-1]}')
print(f'[*] Danh sách các cột thuộc tính    :\n{list(df_benchmark.columns)}')

display(df_benchmark.head(3))


[*] Kích thước South German Credit : 1,000 dòng x 21 cột
[*] Tên cột mục tiêu đối chứng      : credit_risk
[*] Danh sách các cột thuộc tính    :
['status', 'duration', 'credit_history', 'purpose', 'amount', 'savings', 'employment_duration', 'installment_rate', 'personal_status_sex', 'other_debtors', 'present_residence', 'property', 'age', 'other_installment_plans', 'housing', 'number_credits', 'job', 'people_liable', 'telephone', 'foreign_worker', 'credit_risk']


,status,duration,credit_history,purpose,amount,savings,employment_duration,installment_rate,personal_status_sex,other_debtors,present_residence,property,age,other_installment_plans,housing,number_credits,job,people_liable,telephone,foreign_worker,credit_risk
0,no checking account,18,all credits at this bank paid back duly,car (used),1049,unknown/no savings account,< 1 yr,< 20,female : non-single or male : single,none,>= 7 yrs,car or other,21,none,for free,1,skilled employee/official,0 to 2,no,no,good
1,no checking account,9,all credits at this bank paid back duly,others,2799,unknown/no savings account,1 <= ... < 4 yrs,25 <= ... < 35,male : married/widowed,none,1 <= ... < 4 yrs,unknown / no property,36,none,for free,3-Feb,skilled employee/official,3 or more,no,no,good
2,... < 0 DM,12,no credits taken/all credits paid back duly,retraining,841,... < 100 DM,4 <= ... < 7 yrs,25 <= ... < 35,female : non-single or male : single,none,>= 7 yrs,unknown / no property,23,none,for free,1,unskilled - resident,0 to 2,no,no,good
